# Model A - Baseline Time-Series
| Aspek | Isi |
|---|---|
| Target | log return kurs tengah hari berikutnya, `target = ret.shift(-1)` |
| Fitur | 7 fitur kurs |
| Split | kronologis 70/15/15 |
| Model | 14 regresi + LSTM PyTorch, `SEED = 42` |


## 0. Import dan lokasi file

In [12]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    AdaBoostRegressor, GradientBoostingRegressor,
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.6g}")


def find_root(start):
    for cand in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (cand / "data" / "raw" / "bi-usd-rate.csv").exists():
            return cand
    raise FileNotFoundError("Folder proyek tidak ditemukan")


ROOT = find_root(Path.cwd())
RAW_DIR = ROOT / "data" / "raw"
EXPORT_DIR = ROOT / "data" / "export"
OUT_DIR = ROOT / "trial-and-error" / "Ken" / "results"
sys.path.insert(0, str((ROOT / "src").resolve()))
from eval_utils import evaluate, METRICS

START_DATE = pd.Timestamp("2021-09-01")
END_DATE = pd.Timestamp("2026-09-01 23:59:59")
print("Proyek :", ROOT)
print("Output :", OUT_DIR)


Proyek : D:\nlpproject-grestercinta
Output : D:\nlpproject-grestercinta\trial-and-error\Ken\results


## 1. Data kurs

In [13]:
bi_raw = pd.read_csv(RAW_DIR / "bi-usd-rate.csv")
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")

bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)


def clean_numeric(series):
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi = bi_raw.loc[bi_raw["Tanggal"].between(START_DATE, END_DATE, inclusive="both")]
bi = bi.loc[
    bi["Kurs Jual"].notna() & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0) & (bi["Kurs Beli"] > 0)
    & (bi["Kurs Jual"] > bi["Kurs Beli"])
]
bi = bi.sort_values("Tanggal").drop_duplicates(subset=["Tanggal"], keep="last").reset_index(drop=True)
print("Baris kurs valid:", len(bi))


Baris kurs valid: 1202


## 2. Feature engineering

In [14]:
data = bi.sort_values("Tanggal").reset_index(drop=True).copy()
data["Tanggal_dt"] = data["Tanggal"].dt.normalize()
data["kurs_mid"] = (data["Kurs Jual"] + data["Kurs Beli"]) / 2
data["ret"] = np.log(data["kurs_mid"]).diff()
data["target"] = data["ret"].shift(-1)
for lag in range(1, 6):
    data[f"kurs_lag{lag}"] = data["kurs_mid"].shift(lag)
data["kurs_roll_mean5"] = data["kurs_mid"].shift(1).rolling(5).mean()
data["kurs_roll_std5"] = data["kurs_mid"].shift(1).rolling(5).std()

FEAT_A = [f"kurs_lag{lag}" for lag in range(1, 6)] + [
    "kurs_roll_mean5", "kurs_roll_std5"
]
dataset = data.dropna(subset=["ret", "target", *FEAT_A]).reset_index(drop=True)
print("Bentuk dataset:", dataset.shape)


Bentuk dataset: (1196, 18)


## 3. Split kronologis 70/15/15

In [15]:
dataset = dataset.sort_values("Tanggal_dt").reset_index(drop=True)
n = len(dataset)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = dataset.iloc[:train_end].copy()
val = dataset.iloc[train_end:val_end].copy()
test = dataset.iloc[val_end:].copy()
train = train.iloc[:-1].copy()
val = val.iloc[:-1].copy()

assert train["Tanggal_dt"].max() < val["Tanggal_dt"].min() < test["Tanggal_dt"].min()
ytr = train["target"].values
yva = val["target"].values
yte = test["target"].values
for name, part in [("Train", train), ("Val", val), ("Test", test)]:
    print(name, len(part), part["Tanggal_dt"].min().date(), "->", part["Tanggal_dt"].max().date())


Train 836 2021-09-08 -> 2025-02-18
Val 178 2025-02-20 -> 2025-11-20
Test 180 2025-11-24 -> 2026-08-31


## 4. Setup model dan LSTM

In [16]:
MODELS = {
    "LinearRegression": lambda: LinearRegression(),
    "Ridge": lambda: Ridge(),
    "Lasso": lambda: Lasso(),
    "ElasticNet": lambda: ElasticNet(),
    "BayesianRidge": lambda: BayesianRidge(),
    "KNN": lambda: KNeighborsRegressor(),
    "SVR": lambda: SVR(),
    "DecisionTree": lambda: DecisionTreeRegressor(random_state=SEED),
    "RandomForest": lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "ExtraTrees": lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "AdaBoost": lambda: AdaBoostRegressor(random_state=SEED),
    "GradientBoosting": lambda: GradientBoostingRegressor(random_state=SEED),
    "XGBoost": lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
    "LightGBM": lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}

scaler = StandardScaler().fit(train[FEAT_A])
X_tr = scaler.transform(train[FEAT_A])
X_va = scaler.transform(val[FEAT_A])
X_te = scaler.transform(test[FEAT_A])
LOOKBACK = 14


def make_lstm_sequences(features, targets, lookback):
    features = np.asarray(features, dtype=np.float32)
    targets = np.asarray(targets, dtype=np.float32)
    x = np.asarray([features[i - lookback:i]
                    for i in range(lookback, len(features))])
    y = targets[lookback:].copy()
    return x, y


def make_lstm_prediction_sequences(prefix, split_features, lookback):
    prefix = np.asarray(prefix, dtype=np.float32)
    split_features = np.asarray(split_features, dtype=np.float32)
    context = np.vstack([prefix[-lookback:], split_features])
    x = np.asarray([context[i - lookback:i]
                    for i in range(lookback, len(context))])
    return x[-len(split_features):]


class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.head(self.dropout(hidden[-1])).squeeze(-1)


def train_lstm(X_train, y_train, X_val, y_val, X_test):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    x_train, y_train_seq = make_lstm_sequences(X_train, y_train, LOOKBACK)
    y_scaler = StandardScaler().fit(y_train_seq.reshape(-1, 1))
    y_train_scaled = y_scaler.transform(
        y_train_seq.reshape(-1, 1)
    ).ravel().astype(np.float32)

    x_val = make_lstm_prediction_sequences(X_train, X_val, LOOKBACK)
    x_test = make_lstm_prediction_sequences(X_val, X_test, LOOKBACK)
    model = LSTMRegressor(n_features=X_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train_scaled)),
        batch_size=32,
        shuffle=False,
    )
    y_val_scaled = y_scaler.transform(
        np.asarray(y_val).reshape(-1, 1)
    ).ravel().astype(np.float32)
    x_val_tensor = torch.from_numpy(x_val)
    y_val_tensor = torch.from_numpy(y_val_scaled)

    best_state = None
    best_val_loss = float("inf")
    patience_left = 20
    for _ in range(200):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_scaled = model(x_val_tensor).numpy()
            val_loss = float(np.mean((val_scaled - y_val_tensor.numpy()) ** 2))
        if val_loss < best_val_loss - 1e-8:
            best_val_loss = val_loss
            best_state = {key: value.detach().clone()
                          for key, value in model.state_dict().items()}
            patience_left = 20
        else:
            patience_left -= 1
        if patience_left == 0:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = y_scaler.inverse_transform(
            model(x_val_tensor).numpy().reshape(-1, 1)
        ).ravel()
        test_pred = y_scaler.inverse_transform(
            model(torch.from_numpy(x_test)).numpy().reshape(-1, 1)
        ).ravel()
    return val_pred, test_pred


print(f"{len(MODELS)} model sklearn + LSTM")


14 model sklearn + LSTM


## 5. Latih dan evaluasi

In [17]:
rows = []
preds = []

for model_name, make_model in MODELS.items():
    model = make_model().fit(X_tr, ytr)
    for split, X, y, part in [
        ("Val", X_va, yva, val),
        ("Test", X_te, yte, test),
    ]:
        p = model.predict(X)
        rows.append({"Model": model_name, "Split": split,
                     "n_fitur": len(FEAT_A), **evaluate(y, p)})
        preds.append(pd.DataFrame({
            "Model": model_name, "Split": split,
            "date": part["Tanggal_dt"].values,
            "y_true": y, "y_pred": p,
        }))
    print("  selesai:", model_name)

lstm_val, lstm_test = train_lstm(X_tr, ytr, X_va, yva, X_te)
for split, p, y, part in [
    ("Val", lstm_val, yva, val),
    ("Test", lstm_test, yte, test),
]:
    rows.append({"Model": "LSTM", "Split": split,
                 "n_fitur": len(FEAT_A), **evaluate(y, p)})
    preds.append(pd.DataFrame({
        "Model": "LSTM", "Split": split,
        "date": part["Tanggal_dt"].values,
        "y_true": y, "y_pred": p,
    }))
print("  selesai: LSTM")

results = pd.DataFrame(rows)
results["Degenerate"] = results["PredStd"] < 1e-9
preds = pd.concat(preds, ignore_index=True)
print("Total baris metrik:", len(results),
      "| total baris prediksi:", len(preds))


  selesai: LinearRegression
  selesai: Ridge
  selesai: Lasso
  selesai: ElasticNet
  selesai: BayesianRidge
  selesai: KNN
  selesai: SVR
  selesai: DecisionTree
  selesai: RandomForest
  selesai: ExtraTrees
  selesai: AdaBoost
  selesai: GradientBoosting
  selesai: XGBoost
  selesai: LightGBM
  selesai: LSTM
Total baris metrik: 30 | total baris prediksi: 5370


## 6. Hasil dan baseline

In [18]:
for split in ("Val", "Test"):
    print(f"\n=== Model A | {split} ===")
    table = (results[(results["Split"] == split) & ~results["Degenerate"]]
             .sort_values("RMSE")
             .set_index("Model")[["n_fitur", "MAE", "MSE", "RMSE", "DirAcc", "PredStd"]])
    print(table.to_string())

naive = []
for split, y, part in [("Val", yva, val), ("Test", yte, test)]:
    naive.append({"Baseline": "Prediksi 0", "Split": split,
                  **evaluate(y, np.zeros(len(y)))})
    naive.append({"Baseline": "Return kemarin", "Split": split,
                  **evaluate(y, part["ret"].values)})
naive = pd.DataFrame(naive)
print(naive.to_string(index=False))



=== Model A | Val ===
                  n_fitur        MAE         MSE       RMSE   DirAcc     PredStd
Model                                                                           
BayesianRidge           7 0.00250102 1.14832e-05 0.00338869 0.448864  0.00025135
LSTM                    7 0.00248529 1.15067e-05 0.00339216 0.465909  4.2186e-05
Ridge                   7 0.00251092 1.15099e-05 0.00339263 0.460227 0.000353976
LinearRegression        7 0.00252672 1.15704e-05 0.00340153 0.465909 0.000473432
AdaBoost                7 0.00251798  1.1854e-05 0.00344297 0.568182 0.000591661
RandomForest            7 0.00260867 1.19681e-05 0.00345949 0.511364 0.000861584
ExtraTrees              7 0.00260296 1.21402e-05 0.00348428 0.522727 0.000930309
LightGBM                7  0.0027445  1.2737e-05  0.0035689 0.511364  0.00102031
XGBoost                 7 0.00276311 1.31958e-05  0.0036326 0.482955  0.00123167
KNN                     7 0.00284737 1.34228e-05 0.00366371 0.477273  0.00101869
Gradi

## 7. Ekspor CSV

In [19]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT_DIR / "metrics_model_A.csv", index=False)
preds[preds["Split"] == "Val"].reset_index(drop=True).to_csv(
    OUT_DIR / "predictions_val_model_A.csv", index=False
)
preds[preds["Split"] == "Test"].reset_index(drop=True).to_csv(
    OUT_DIR / "predictions_test_model_A.csv", index=False
)
naive.to_csv(OUT_DIR / "naive_baselines_model_A.csv", index=False)

export_cols = ["Tanggal_dt", "kurs_mid", "ret", "target", *FEAT_A]
export_ds = pd.concat([
    train[export_cols].assign(split="train"),
    val[export_cols].assign(split="val"),
    test[export_cols].assign(split="test"),
], ignore_index=True)
export_ds.to_parquet(OUT_DIR / "dataset_model_A.parquet", index=False)

metadata = {
    "model": "A",
    "deskripsi": "Baseline time-series murni, tanpa fitur NLP",
    "target": "log return kurs tengah hari berikutnya",
    "fitur": FEAT_A,
    "n_fitur": len(FEAT_A),
    "models": [*MODELS.keys(), "LSTM"],
    "seed": SEED,
    "lstm": {"lookback": LOOKBACK, "hidden_size": 32, "epochs_max": 200,
             "early_stopping_patience": 20},
    "metrik": METRICS,
}
(OUT_DIR / "metadata_model_A.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Keluaran tersimpan di:", OUT_DIR.relative_to(ROOT))


Keluaran tersimpan di: trial-and-error\Ken\results
